# Evaluating Agentic AI for Ontology Curation

Systematic evaluation of AI coding agents on biomedical ontology editing tasks.
We compare agent **harnesses** (Codex, OpenCode, Pi) on the same model (GPT-5.5)
to isolate the effect of runtime infrastructure on agent performance.

In [ ]:
import os
while not os.path.exists('analysis/scores.tsv'):
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from scipy import stats
from pathlib import Path
from ai4c_scribe.scoring import load_agents_config, EVAL_REPOS

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
pd.set_option('display.precision', 3)

# Load from pre-computed TSV (fast, no API calls)
df = pd.read_csv('analysis/scores.tsv', sep='\t')
df['f1'] = pd.to_numeric(df['f1'], errors='coerce')
print(f'Loaded {len(df)} scored runs from analysis/scores.tsv')
print(f'Agents: {sorted(df["agent"].unique())}')

---
## 1. Experiment Overview

### Ontologies under test

In [ ]:
ont_info = []
for ont, cfg in EVAL_REPOS.items():
    n_cases = df[df['ontology']==ont]['case'].nunique()
    n_runs = len(df[df['ontology']==ont])
    ont_info.append({'ontology': ont, 'eval_repo': f'ai4curation/{cfg["eval_repo"]}',
                     'source': cfg['source_repo'], 'format': cfg['metadiff_config'],
                     'cases': n_cases, 'runs': n_runs})
pd.DataFrame(ont_info).set_index('ontology')

### Agents tested

An **agent** is the complete configuration tuple: harness + model + config + reasoning effort.
Logical handles (e.g., `std_codex_g55`) abstract away repo-specific version tags.

In [ ]:
# Load agents from all ontologies
all_agents = {}
for ont in EVAL_REPOS:
    agents = load_agents_config(ont)
    for handle, cfg in agents.items():
        if handle not in all_agents:
            all_agents[handle] = cfg

agents_df = pd.DataFrame([
    {'agent': h, 'runtime': c['runtime'], 'model': c['model'],
     'skills': c.get('skills', True), 'category': c.get('category', 'standard'),
     'description': c['description']}
    for h, c in all_agents.items()
])
agents_df.set_index('agent')

### Test cases by type and difficulty

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df.drop_duplicates('case').groupby('case_type').size().sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Unique Cases by Task Type')
axes[0].set_xlabel('Count')
df.drop_duplicates('case').groupby('difficulty').size().reindex(['simple','medium','hard']).plot.barh(ax=axes[1], color='darkorange')
axes[1].set_title('Unique Cases by Difficulty')
axes[1].set_xlabel('Count')
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_cases_breakdown.png', dpi=150)
plt.show()

print(f'Total unique cases: {df["case"].nunique()}')
print(f'Total scored runs: {len(df)}')
print(f'Runs with F1 > 0: {(df["f1"]>0).sum()} ({(df["f1"]>0).mean()*100:.0f}%)')

### Experiment design

The primary experiment compares three harnesses (Codex, OpenCode, Pi) on GPT-5.5
with the standard config for each ontology. This isolates the harness effect:
same model, same skills, same cases, different runtime infrastructure.

Secondary comparisons include:
- **Skills ablation**: standard vs no-skills config (GO, Mondo)
- **Model tier**: GPT-5.5 vs GPT-5.4 vs Claude Sonnet/Haiku
- **Ontology difficulty**: GO (mature skills) vs CL/Uberon/Mondo (less mature)

---
## 2. Main Result: Harness Comparison

Paired comparison of Codex vs OpenCode on gpt-5.5 across shared cases.

In [ ]:
# Filter to standard agents only
std = df[df['agent'].str.startswith('std_')].copy()
print(f'Standard config runs: {len(std)}')
print(f'By agent: {std["agent"].value_counts().to_dict()}')

In [ ]:
# Paired comparison
codex = std[std['agent']=='std_codex_g55'].groupby('case')['f1'].mean()
opencode = std[std['agent']=='std_opencode_g55'].groupby('case')['f1'].mean()
shared = codex.index.intersection(opencode.index)
c, o = codex.loc[shared].values, opencode.loc[shared].values
n = len(c)
diff = o - c

t_stat, p_val = stats.ttest_rel(o, c)
rng = np.random.default_rng(42)
boot = np.array([diff[rng.integers(0, n, n)].mean() for _ in range(10000)])
ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
d = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else 0

print(f'Paired: std_codex_g55 vs std_opencode_g55')
print(f'  n = {n} shared cases')
print(f'  Codex:    {c.mean():.3f} ± {c.std():.3f}')
print(f'  OpenCode: {o.mean():.3f} ± {o.std():.3f}')
print(f'  Diff:     {diff.mean():+.3f}')
print(f'  t({n-1}) = {t_stat:.3f}, p = {p_val:.4f}')
print(f'  95% CI:   [{ci_lo:.3f}, {ci_hi:.3f}]')
print(f'  Cohen d:  {d:.3f}')
print(f'  Significant: {"YES" if p_val < 0.05 else "NO"}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
ax = axes[0]
ax.scatter(c, o, alpha=0.5, s=40)
ax.plot([0,1.05],[0,1.05],'k--',alpha=0.3)
ax.set_xlim(-0.05,1.05); ax.set_ylim(-0.05,1.05)
ax.set_xlabel('std_codex_g55 F1'); ax.set_ylabel('std_opencode_g55 F1')
ax.set_title(f'Paired (n={n}), p={p_val:.3f}')

ax = axes[1]
ax.hist(diff, bins=20, alpha=0.7, edgecolor='black')
ax.axvline(0, color='red', ls='--'); ax.axvline(diff.mean(), color='blue')
ax.axvspan(ci_lo, ci_hi, alpha=0.15, color='blue')
ax.set_xlabel('F1 diff (OpenCode − Codex)'); ax.set_title('Differences')

ax = axes[2]
for i, d_level in enumerate(['simple','medium','hard']):
    sub = std[std['difficulty']==d_level]
    cs = sub[sub['agent']=='std_codex_g55'].groupby('case')['f1'].mean()
    os_ = sub[sub['agent']=='std_opencode_g55'].groupby('case')['f1'].mean()
    sh = cs.index.intersection(os_.index)
    if len(sh)>=3:
        diffs = (os_.loc[sh] - cs.loc[sh]).values
        ax.boxplot([diffs], positions=[i], widths=0.6)
ax.set_xticks(range(3)); ax.set_xticklabels(['simple','medium','hard'])
ax.axhline(0, color='red', ls='--', alpha=0.5)
ax.set_ylabel('F1 diff'); ax.set_title('By Difficulty')
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_harness_paired.png', dpi=150)
plt.show()

**Finding**: OpenCode outperforms Codex by a small but significant margin on the same
model. The effect is driven by medium-difficulty cases where skill activation and
tool orchestration matter most.

---
## 3. Performance by Ontology

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
std_nonzero = std[std['f1'] > 0]
sns.boxplot(data=std_nonzero, x='ontology', y='f1', hue='agent', ax=ax)
ax.set_title('F1 by Ontology and Agent (standard config, non-zero only)')
ax.set_ylabel('Metadiff F1'); ax.set_xlabel('')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('analysis/notebooks/fig_by_ontology.png', dpi=150)
plt.show()

---
## 4. Summary

In [ ]:
print(f'Dataset: {df["case"].nunique()} cases across {df["ontology"].nunique()} ontologies')
print(f'Runs: {len(df)} total, {(df["f1"]>0).sum()} non-zero ({(df["f1"]>0).mean()*100:.0f}%)')
print(f'Overall mean F1: {df["f1"].mean():.3f} (non-zero: {df[df["f1"]>0]["f1"].mean():.3f})')
print()
print('Agent summary (standard config only):')
print(std.groupby('agent')['f1'].agg(['count','mean','std']).round(3).sort_values('mean', ascending=False))